# nb00 - Extract: Medi-Cal Managed Care Enrollment

**Purpose:** pull the latest Medi-Cal Managed Care Enrollment Report from the CalHHS Open Data Portal, save a raw copy, and run a first profile of Los Angeles County.

**Why the CKAN API and not the download button:** the CSV filename changes on every monthly refresh. The CKAN `package_show` endpoint always resolves to the current file, so this notebook does not break when DHCS publishes a new month.

**Dataset:** https://data.chhs.ca.gov/dataset/medi-cal-managed-care-enrollment-report

**Grain:** one row per Enrollment Month, Plan Type, County, Plan Name.

**Pipeline:** nb00 (extract, this file) -> nb01 (clean) -> nb02 (SQL and pandas parity).

In [1]:
import json
from datetime import date
from pathlib import Path
import pandas as pd
import requests

CKAN_BASE = 'https://data.chhs.ca.gov/api/3/action'
DATASET_SLUG = 'medi-cal-managed-care-enrollment-report'

DATA_DIR = Path('..') / 'data'
RAW_DIR = DATA_DIR / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)
print('Raw data folder:', RAW_DIR.resolve())

Raw data folder: /Users/trinidadcisneros/Documents/Development/trinidadcisneros.github.io/folders/ds_blogs/projects/tableau/tableau_la_market_share/data/raw


## 1. Resolve the current CSV via the CKAN API

In [2]:
resp = requests.get(f'{CKAN_BASE}/package_show', params={'id': DATASET_SLUG}, timeout=60)
resp.raise_for_status()
pkg = resp.json()['result']

csv_resources = [r for r in pkg['resources'] if r.get('format', '').upper() == 'CSV']
resource = csv_resources[0]
print('Dataset title :', pkg['title'])
print('Resource name :', resource['name'])
print('Last modified :', resource.get('last_modified'))
print('Download URL  :', resource['url'])

Dataset title : Medi-Cal Managed Care Enrollment Report
Resource name : Medi-Cal Managed Care Enrollment Report
Last modified : 2026-07-01T22:03:04.335947
Download URL  : https://data.chhs.ca.gov/dataset/c6ccef54-e7a9-4ebd-b79a-850b72c4dd8c/resource/95358a7a-2c9d-41c6-a0e0-405a7e5c5f18/download/open-data-portal-managed-care-enrollment-count-june-2026.csv


## 2. Download and save a dated raw copy

In [3]:
download = requests.get(resource['url'], timeout=300)
download.raise_for_status()

raw_path = RAW_DIR / f"medi_cal_mc_enrollment_raw_{date.today().isoformat()}.csv"
raw_path.write_bytes(download.content)
print(f'Saved {raw_path.name} ({raw_path.stat().st_size/1_048_576:.1f} MB)')

# log what was pulled, for reproducibility
(DATA_DIR / 'extraction_log.json').write_text(json.dumps({
    'dataset': pkg['title'],
    'resource_name': resource['name'],
    'resource_url': resource['url'],
    'last_modified': resource.get('last_modified'),
    'downloaded': date.today().isoformat(),
    'raw_file': raw_path.name,
}, indent=2))

Saved medi_cal_mc_enrollment_raw_2026-07-22.csv (2.5 MB)


465

## 3. First profile

The file mixes every managed care product line statewide. Here is what one row looks like and what the columns contain.

In [4]:
raw = pd.read_csv(raw_path, dtype=str)
raw.columns = [c.strip() for c in raw.columns]
print('shape:', raw.shape)
print('columns:', list(raw.columns))
print()
for c in ['Enrollment Month', 'Plan Type', 'County']:
    u = raw[c].dropna().unique()
    print(f'{c}: {len(u)} distinct  (range {min(u)} .. {max(u)})' if c=='Enrollment Month'
          else f'{c}: {len(u)} distinct -> {sorted(u)[:12]}')

shape: (31178, 8)
columns: ['Enrollment Month', 'Plan Type', 'County', 'Plan Name', 'Count of Enrollees', 'Count of Enrollees Annotation Code', 'Count of Enrollees Annotation Description', 'Unnamed: 7']

Enrollment Month: 234 distinct  (range 2007-01 .. 2026-06)
Plan Type: 20 distinct -> ['CCS Demonstration', 'COHS', 'Cal MediConnect', 'Commercial Plan (2 Plan)', 'County Organized Health Systems', 'Dental', 'Geographic Managed Care', 'Imperial Model', 'Local Initiative (2 Plan)', 'PACE', 'Prepaid Health Plan', 'Primary Care Case Management']
County: 59 distinct -> ['Alameda', 'Alpine', 'Amador', 'Butte', 'Calaveras', 'Colusa', 'Contra Costa', 'Del Norte', 'El Dorado', 'Fresno', 'Glenn', 'Humboldt']


## 4. Los Angeles County, latest month

A first look at the market this project is about. Note the plan name variants and the mix of medical and non-medical lines, both of which nb01 handles.

In [5]:
raw['n'] = pd.to_numeric(raw['Count of Enrollees'].str.replace(',', '').str.strip(), errors='coerce')
for c in ['Plan Name', 'Plan Type', 'County']:
    raw[c] = raw[c].str.strip()

la = raw[raw.County == 'Los Angeles']
latest = la['Enrollment Month'].max()
print('Latest month:', latest)
print('\nLA plan types present:', sorted(la['Plan Type'].unique()))
print('\nTop LA plans this month (all lines, before cleaning):')
cur = la[la['Enrollment Month'] == latest].groupby('Plan Name').n.sum().sort_values(ascending=False)
print(cur.head(12).apply(lambda v: f'{int(v):,}').to_string())

Latest month: 2026-06

LA plan types present: ['Cal MediConnect', 'Commercial Plan (2 Plan)', 'Dental', 'Local Initiative (2 Plan)', 'PACE', 'Primary Care Case Management', 'Primary Care Case Mgmt', 'SCAN', 'Special Project', 'Two-Plan']

Top LA plans this month (all lines, before cleaning):
Plan Name
L.A. Care Health Plan/Los Angeles               2,102,545
Health Net Community Solutions/Los Angeles      1,079,978
Kaiser Permanente/Los Angeles                     339,151
Health Net Comm Solutions LA                      191,244
Liberty Dental Plan of CA/Los Angeles              88,013
California Dental Network/Los Angeles              18,485
SCAN Health Plan/Los Angeles                       14,680
AltaMed Health Sr. Buena Care/Los Angeles           4,827
Pacific PACE/Los Angeles                            1,559
LA Coast PACE/Los Angeles                           1,049
AIDS Healthcare Foundation/Los Angeles                854
Brandman Centers for Senior Care/Los Angeles          439
